PROJET BIG DATA 

Sujet : Marché Immobilier
Objectif : prédire le prix des biens
source choisie
colonnes qu’on veut extraire
variable cible = prix
variables explicatives = surface, ville, nombre de pièces, type de bien, description, etc.

choisssons Studapart + ? car 
Studapart car pas bloque 

Dans cette partie de code, notre but est de recuperer des infos par le site studapart et d'avoir notre bdd

In [3]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from bs4 import BeautifulSoup
import pandas as pd
import json
import time
import re
import os
import csv

url = "https://www.studapart.com/fr/s/?city%5B0%5D=Paris&city%5B1%5D=Lyon&city%5B2%5D=bordeaux&city%5B3%5D=Marseille&city%5B4%5D=toulouse&city%5B5%5D=nantes&city%5B6%5D=nice&city%5B7%5D=lille&city%5B8%5D=montpellier&city%5B9%5D=rennes"

os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)

VILLES = [
    "paris", "lyon", "bordeaux", "marseille", "toulouse",
    "nantes", "nice", "lille", "montpellier", "rennes",
    "colombes", "puteaux", "créteil", "creteil", "ivry",
    "cachan", "sevres", "sèvres", "noisy", "villejuif"
]

TYPES_LOGEMENT = [
    "Logement en résidence",
    "Logement entier",
    "Logement chez l'habitant",
    "Logement T2",
    "Colocation",
    "Studio",
    "Appartement",
    "Maison"
]

MOTS_A_EXCLURE = [
    "comment fonctionne",
    "questions fréquentes",
    "faq",
    "guide propriétaires",
    "bon à savoir",
    "budget moyen",
    "vous êtes propriétaire",
    "toutes nos",
    "louer logement étudiant",
    "recherche en cours",
    "pas de résultats",
    "a la recherche du logement étudiant",
    "le prix moyen",
    "politique zéro ghosting",
    "garantie studapart",
    "1er loyer assuré",
    "type de logement",
    "quartier latin",
    "saint-germain-des-prés",
    "montparnasse",
    "belleville",
    "bibliothèque / ivry"
]

def clean_text(text):
    if text is None:
        return None
    text = str(text)
    text = text.replace("\n", " ").replace("\r", " ").replace("\t", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text


def extract_prix(text):
    if not text:
        return None
    
    # exemples : 760€ / 1 248€ / 1469€
    match = re.search(r'(\d[\d\s]*)\s?€', text)
    if match:
        try:
            return int(match.group(1).replace(" ", ""))
        except:
            return None
    return None


def extract_surface(text):
    if not text:
        return None
    
    match = re.search(r'(\d+(?:[.,]\d+)?)\s*m²', text.lower())
    if match:
        try:
            return float(match.group(1).replace(",", "."))
        except:
            return None
    return None


def extract_nb_pieces(text):
    if not text:
        return None
    
    # pièces ou chambres
    match = re.search(r'(\d+)\s*(pièces?|chambres?)', text.lower())
    if match:
        try:
            return int(match.group(1))
        except:
            return None
    return None


def extract_ville(text):
    if not text:
        return None
    
    txt = text.lower()
    for ville in VILLES:
        if ville in txt:
            # uniformisation
            mapping = {
                "paris": "Paris",
                "lyon": "Lyon",
                "bordeaux": "Bordeaux",
                "marseille": "Marseille",
                "toulouse": "Toulouse",
                "nantes": "Nantes",
                "nice": "Nice",
                "lille": "Lille",
                "montpellier": "Montpellier",
                "rennes": "Rennes",
                "colombes": "Colombes",
                "puteaux": "Puteaux",
                "créteil": "Créteil",
                "creteil": "Créteil",
                "ivry": "Ivry",
                "cachan": "Cachan",
                "sevres": "Sèvres",
                "sèvres": "Sèvres",
                "noisy": "Noisy",
                "villejuif": "Villejuif",
            }
            return mapping.get(ville, ville.capitalize())
    return None


def extract_type_logement(text):
    if not text:
        return None
    
    txt = text.lower()
    for t in TYPES_LOGEMENT:
        if t.lower() in txt:
            return t
    return None


def extract_titre(text, type_logement=None):
    if not text:
        return None
    
    t = clean_text(text)

    # enlève quelques mots parasites
    for mot in ["360°", "Promotion en cours", "Pro", "Réservable à la chambre"]:
        t = t.replace(mot, "").strip()

    if type_logement and type_logement in t:
        parts = t.split(type_logement, 1)
        if len(parts) > 1:
            t = parts[1].strip()

    # coupe avant le premier prix
    t = re.split(r'\d[\d\s]*\s?€', t)[0].strip()

    # coupe si le titre est beaucoup trop long
    return t[:180] if t else None


def looks_like_listing(text):
    if not text:
        return False

    txt = text.lower()

    if "€" not in text:
        return False

    if len(text) < 40:
        return False

    if any(mot in txt for mot in MOTS_A_EXCLURE):
        return False

    if not any(t.lower() in txt for t in TYPES_LOGEMENT):
        return False

    return True

options = Options()
options.add_argument("--start-maximized")
options.add_argument("--disable-blink-features=AutomationControlled")

driver = webdriver.Chrome(options=options)

try:
    driver.get(url)
    time.sleep(8)

    # scroll pour charger plus d'annonces
    for _ in range(6):
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(2)

    html = driver.page_source

finally:
    driver.quit()


soup = BeautifulSoup(html, "html.parser")
elements = soup.find_all(True)

print("Nombre total d'éléments HTML :", len(elements))

annonces = []

for el in elements:
    texte = clean_text(el.get_text(" ", strip=True))

    if not looks_like_listing(texte):
        continue

    prix = extract_prix(texte)
    type_logement = extract_type_logement(texte)

    if prix is None or type_logement is None:
        continue

    surface = extract_surface(texte)
    nb_pieces = extract_nb_pieces(texte)
    ville = extract_ville(texte)
    titre = extract_titre(texte, type_logement=type_logement)

    annonces.append({
        "ville": ville,
        "prix": prix,
        "type_logement": type_logement,
        "surface": surface,
        "nb_pieces": nb_pieces,
        "description": texte,
        "titre": titre,
        "source": "studapart"
    })

print("Nombre d'annonces brutes détectées :", len(annonces))


df_raw = pd.DataFrame(annonces)

if df_raw.empty:
    print("Aucune annonce extraite. Il faudra ajuster les filtres.")
else:
    df_raw["description"] = df_raw["description"].apply(clean_text)
    df_raw["titre"] = df_raw["titre"].apply(clean_text)

print("\nAperçu brut :")
print(df_raw.head(10))
print("\nValeurs manquantes brut :")
print(df_raw.isnull().sum())


df_raw.to_csv(
    "data/raw/studapart_raw.csv",
    sep=";",
    index=False,
    encoding="utf-8-sig",
    quoting=csv.QUOTE_ALL
)

with open("data/raw/studapart_raw.json", "w", encoding="utf-8") as f:
    json.dump(df_raw.to_dict(orient="records"), f, ensure_ascii=False, indent=2)

print("\n✅ Fichiers bruts sauvegardés :")
print("- data/raw/studapart_raw.csv")
print("- data/raw/studapart_raw.json")


df_model = df_raw.copy()

# enlève prix trop faibles / aberrants
df_model = df_model[df_model["prix"].notna()]
df_model = df_model[df_model["prix"] > 200]

# enlève titres trop vides
df_model = df_model[df_model["titre"].notna()]
df_model = df_model[df_model["titre"].str.len() > 3]

# enlève descriptions trop courtes
df_model = df_model[df_model["description"].notna()]
df_model = df_model[df_model["description"].str.len() > 20]

# type_bien = même chose que type_logement pour l'instant
df_model["type_bien"] = df_model["type_logement"]

# suppression des doublons
df_model = df_model.drop_duplicates(subset=["prix", "titre"]).reset_index(drop=True)

# réorganisation colonnes
df_model = df_model[
    ["prix", "ville", "nb_pieces", "surface", "type_bien", "type_logement", "description", "titre", "source"]
]

print("\nAperçu dataset propre :")
print(df_model.head(10))
print("\nTaille finale :", df_model.shape)
print("\nValeurs manquantes dataset propre :")
print(df_model.isnull().sum())


df_model.to_csv(
    "data/processed/studapart_model.csv",
    sep=";",
    index=False,
    encoding="utf-8-sig",
    quoting=csv.QUOTE_ALL
)

with open("data/processed/studapart_model.json", "w", encoding="utf-8") as f:
    json.dump(df_model.to_dict(orient="records"), f, ensure_ascii=False, indent=2)

print("\n✅ Fichiers propres sauvegardés :")
print("- data/processed/studapart_model.csv")
print("- data/processed/studapart_model.json")

Nombre total d'éléments HTML : 8771
Nombre d'annonces brutes détectées : 317

Aperçu brut :
      ville  prix          type_logement  surface  nb_pieces  \
0     Paris   690  Logement en résidence    430.0       15.0   
1     Paris   690  Logement en résidence    430.0       15.0   
2     Paris   690  Logement en résidence    430.0       15.0   
3     Paris   690  Logement en résidence    430.0       15.0   
4     Paris   690  Logement en résidence    430.0       15.0   
5     Paris   690  Logement en résidence    430.0       15.0   
6     Paris   690  Logement en résidence    430.0       15.0   
7  Toulouse   690  Logement en résidence      NaN        NaN   
8  Toulouse   690  Logement en résidence      NaN        NaN   
9  Toulouse   690  Logement en résidence      NaN        NaN   

                                         description  \
0  Colocations à Paris uniquement Colocations à P...   
1  Colocations à Paris uniquement Colocations à P...   
2  Colocations à Paris uniquement C